In [3]:
#get all the tables:
from db_setup import conn, currencyexchange, customer, date, product, sales, store
#instead of saving csv files in sql and then importing one by one
import pandas as pd

order_enriched=pd.read_parquet(r"C:\Users\amyre\Desktop\Luke Barousse SQL Course\SQL + Contoso\saved_files\order_enriched.parquet")


0- Cleanup our dates

In [4]:
#checked with df.info() and the date columns are strings. do converting all of them (even those i havent checked):

sales["orderdate"] = pd.to_datetime(sales["orderdate"])
sales["deliverydate"] = pd.to_datetime(sales["deliverydate"])
customer["birthday"] = pd.to_datetime(customer["birthday"])
customer["startdt"] = pd.to_datetime(customer["startdt"])
customer["enddt"] = pd.to_datetime(customer["enddt"])
store["opendate"] = pd.to_datetime(store["opendate"])
store["closedate"] = pd.to_datetime(store["closedate"])
date["date"] = pd.to_datetime(date["date"])
currencyexchange["date"] = pd.to_datetime(currencyexchange["date"])



In [ ]:
customer_summary = (
    order_enriched
    .groupby("customerkey")
    .agg(
        first_purchase_date=("orderdate", "min"),
        last_purchase_date=("orderdate", "max"),
        order_count=("orderkey", "nunique"),
        total_units=("total_units", "sum"),
        lifetime_revenue=("order_revenue", "sum"),
        lifetime_cost=("order_cost", "sum"),
        lifetime_profit=("order_profit", "sum")
    )
    .reset_index()
)

customer_summary = customer_summary.merge(
    customer[["customerkey", "countryfull", "gender", "occupation", "birthday"]],
    on="customerkey",
    how="inner"
)

customer_summary = customer_summary.rename(columns={"countryfull": "customer_country"})

#add the metrics , just like in sql:

#avgs revenue, profit, margin per order:
customer_summary["avg_order_value"] = (
    customer_summary["lifetime_revenue"] / customer_summary["order_count"]
)

customer_summary["avg_order_profit"] = (
    customer_summary["lifetime_profit"] / customer_summary["order_count"]
)

customer_summary["profit_margin"] = (
    customer_summary["lifetime_profit"] / customer_summary["lifetime_revenue"]
)

#last dataset date and recency days and frequency+monetary (RFM):
dataset_last_date = order_enriched["orderdate"].max()

customer_summary["recency_days"] = (
    dataset_last_date - customer_summary["last_purchase_date"]
).dt.days

customer_summary["customer_lifespan_days"] = (
    customer_summary["last_purchase_date"] - customer_summary["first_purchase_date"]
).dt.days

customer_summary["cohort_month"] = (
    customer_summary["first_purchase_date"].dt.to_period("M").dt.to_timestamp()
)

customer_summary["cohort_year"] = customer_summary["first_purchase_date"].dt.year

customer_summary["age_at_first_purchase"]=(
    customer_summary["first_purchase_date"].dt.year - customer_summary["birthday"].dt.year
    - (
        (customer_summary["first_purchase_date"].dt.month < customer_summary["birthday"].dt.month) |
        (
            (customer_summary["first_purchase_date"].dt.month==customer_summary["birthday"].dt.month) &
            (customer_summary["first_purchase_date"].dt.day < customer_summary["birthday"].dt.day)
        )
    ).astype(int)
)

customer_summary["frequency"] = customer_summary["order_count"]
customer_summary["monetary"] = customer_summary["lifetime_revenue"]

In [6]:
customer_summary

,customerkey,first_purchase_date,last_purchase_date,order_count,total_units,lifetime_revenue,lifetime_cost,lifetime_profit,customer_country,gender,...,avg_order_value,avg_order_profit,profit_margin,recency_days,customer_lifespan_days,cohort_month,cohort_year,age_at_first_purchase,frequency,monetary
0,15,2021-03-08,2021-03-08,1,5,1299.708307,635.866694,663.841613,Australia,male,...,1299.708307,663.841613,0.510762,1139,0,2021-03-01,2021,55,1,1299.708307
1,180,2018-07-28,2023-08-28,2,6,1103.346104,554.979312,548.366791,Australia,male,...,551.673052,274.183396,0.497003,236,1857,2018-07-01,2018,63,2,1103.346104
2,185,2019-06-01,2019-06-01,1,3,666.453820,386.105525,280.348295,Australia,female,...,666.453820,280.348295,0.420657,1785,0,2019-06-01,2019,39,1,666.453820
3,243,2016-05-19,2016-05-19,1,5,148.903542,82.504155,66.399387,Australia,female,...,148.903542,66.399387,0.445922,2893,0,2016-05-01,2016,62,1,148.903542
4,387,2018-12-21,2023-11-16,3,26,2341.268364,1123.732736,1217.535628,Australia,female,...,780.422788,405.845209,0.520032,156,1791,2018-12-01,2018,33,3,2341.268364
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49482,2099619,2018-09-11,2020-07-10,4,25,6709.935970,2766.376800,3943.559170,United States,female,...,1677.483992,985.889792,0.587719,1380,668,2018-09-01,2018,50,4,6709.935970
49483,2099656,2023-05-11,2024-02-06,4,46,10404.677800,4919.090000,5485.587800,United States,male,...,2601.169450,1371.396950,0.527223,74,271,2023-05-01,2023,77,4,10404.677800
49484,2099697,2022-09-13,2022-09-13,1,5,38.201100,17.830000,20.371100,United States,male,...,38.201100,20.371100,0.533260,585,0,2022-09-01,2022,55,1,38.201100
49485,2099711,2016-08-13,2017-08-14,2,7,6008.670000,3018.760000,2989.910000,United States,female,...,3004.335000,1494.955000,0.497599,2441,366,2016-08-01,2016,75,2,6008.670000


In [7]:
#saving the views as parquet files to be later used on other notebooks:

customer_summary.to_parquet(r"C:\Users\amyre\Desktop\Luke Barousse SQL Course\SQL + Contoso\saved_files\customer_summary.parquet", index=False)